# Cleaning Validation & Data Quality Audit — CatalogIQ

**Version 2.** Rebuilt after review. The first version reported results that its
own checks did not support; every change below exists because something it said
was wrong.

| What review found | What changed here |
|---|---|
| Matching totals is not matching rows | §1 compares the row **sets**, not the counts |
| Only 76,219 of 83,938 retained rows were compared | §7 compares all of them and prints the coverage |
| Shifted-column checks skipped quoted records | §9 includes them and validates each column's domain |
| Rare labels with stray spaces were excluded by rule | §6 evaluates them on evidence instead |
| Only training was audited | every check runs on **training and target** |
| Run directory auto-detected | §0 selects one run explicitly and refuses to mix |
| Conclusions overstated | §9 and §10 state what the evidence supports, no more |

**A standing rule for this notebook.** A PASS must be produced by a check that
actually ran over the full population it claims to cover. Where a check cannot
cover everything, it says so and reports BLOCKED or WARN rather than PASS.

## 0. Setup — one explicit run

The run directory is chosen explicitly. Auto-detection across `data/processed/`
can pair a candidate file from one run with a decision table from another, and
the resulting numbers would reconcile against nothing.

Set `RUN_NAME` to the directory under `data/processed/`. Leave it `None` to use
the newest, which is printed and must be confirmed against the one being
audited.

In [ ]:
import pandas as pd, re, json
from pathlib import Path

pd.set_option("display.max_rows", 250); pd.set_option("display.width", 170)

# The run directory under data/processed/ to audit. Set it: leaving it None lets
# the notebook pick the newest, which is how artifacts from two runs get mixed.
RUN_NAME = "my_cleaning_run"

CATALOGIQ = None
try:
    import sys
    for _p in (Path.cwd(), *Path.cwd().parents):
        if (_p / "src" / "catalogiq" / "paths.py").is_file():
            sys.path.insert(0, str(_p / "src")); break
    from catalogiq import paths as CATALOGIQ
except Exception:
    pass

def project_root(start="."):
    if CATALOGIQ is not None: return CATALOGIQ.ROOT_PATH
    p = Path(start).resolve()
    for _ in range(8):
        if (p/"data"/"provided").is_dir() or (p/".git").exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

ROOT = project_root()
DATA = CATALOGIQ.DATA_PATH if CATALOGIQ is not None else ROOT/"data"
PROVIDED, PROCESSED = DATA/"provided", DATA/"processed"
print("project root:", ROOT)

runs = sorted([d for d in PROCESSED.glob("*") if d.is_dir() and (d/"summary.json").is_file()],
              key=lambda d: d.stat().st_mtime, reverse=True) if PROCESSED.is_dir() else []
if not runs:
    raise SystemExit(f"No run directory with a summary.json under {PROCESSED}. "
                     f"Run: python -m catalogiq --mode integrated --output-dir data/processed/<name>")
RUN = (PROCESSED/RUN_NAME) if RUN_NAME else runs[0]
if not (RUN/"summary.json").is_file():
    print(f"RUN_NAME is set to {RUN_NAME!r} but {RUN} has no summary.json.")
    print(f"Available runs: {[d.name for d in runs]}")
    raise SystemExit("Set RUN_NAME to one of the runs listed above.")

print("runs available :", [d.name for d in runs])
print("AUDITING RUN   :", RUN.name)
if RUN_NAME is None:
    print("  ! RUN_NAME is None, so the newest run was chosen automatically.")
    print("  ! Set it explicitly - this is how files from two runs get mixed.")

SUMMARY = json.loads((RUN/"summary.json").read_text())
print("policy version :", SUMMARY.get("policy_version", "unknown"))

TARGETS = ["Mnfr","Brand","Platform","Segment","Sub-Segment","TargetAgeGroup"]
NUMERIC = ["ProductRating","ProductReviewsCount","XRatXRev","ReviewsCount"]
URLCOLS = ["ProductUrl","ProductImageUrl"]

DATASETS = {}
for tag, rawname, candname in [
        ("training","Selfcare_Training_data (1).csv","training_candidate.csv"),
        ("target",  "Selfcare_Target_data (1).csv",  "target_candidate.csv")]:
    rp, cp = PROVIDED/rawname, RUN/candname
    if not rp.is_file(): print(f"  {tag}: raw missing at {rp}"); continue
    if not cp.is_file(): print(f"  {tag}: candidate missing at {cp}"); continue
    raw  = pd.read_csv(rp, dtype="string", keep_default_na=False, low_memory=False)
    cand = pd.read_csv(cp, dtype="string", keep_default_na=False, low_memory=False)
    raw["source_row"] = (raw.index + 1).astype(str)   # summary.json: source_row_base = 1
    DATASETS[tag] = {"raw": raw, "cand": cand, "run": RUN}
    print(f"  {tag:9s} raw {len(raw):>7,}  candidate {len(cand):>7,}")

if not DATASETS:
    raise SystemExit("No dataset could be loaded. Put the PO's CSVs in data/provided/.")

RD = pd.read_csv(RUN/"row_decisions.csv", dtype=str, low_memory=False)
LABEL_CHANGES = (pd.read_csv(RUN/"label_changes.csv", dtype=str)
                 if (RUN/"label_changes.csv").is_file() else pd.DataFrame())
print(f"\nrow_decisions {len(RD):,} rows   label_changes {len(LABEL_CHANGES):,} rows")

### 0.1 Verdict recorder

Every verdict carries the dataset it applies to and the population the check
covered. A PASS whose coverage is less than the population it claims is
downgraded automatically.

In [ ]:
VERDICTS = []

def verdict(dataset, criterion, status, detail, covered=None, population=None):
    if status == "PASS" and covered is not None and population and covered < population:
        status = "WARN"
        detail = (f"Check covered {covered:,} of {population:,} rows, so this "
                  f"cannot be reported as PASS. " + detail)
    VERDICTS.append({"dataset":dataset,"criterion":criterion,"status":status,
                     "covered":covered,"population":population,"detail":detail})
    bar = {"PASS":"","WARN":"!  ","FAIL":"!! ","BLOCKED":"-- "}[status]
    cov = f"  [{covered:,}/{population:,}]" if covered is not None and population else ""
    print(f"\n[{dataset}] {criterion}: {bar}{status}{cov}")
    for line in str(detail).split("\n"): print("   " + line)

def blank(s):
    if isinstance(s, pd.DataFrame):
        return s.apply(lambda col: col.str.strip().str.lower().isin(["", "null"]))
    return s.str.strip().str.lower().isin(["", "null"])
def live(s):   return s[~blank(s)]

## 1. Row counts, and the same rows

Counts reconciling is necessary and not sufficient. Two disjoint sets of 614
rows also reconcile. This compares the **sets**: the rows absent from the
candidate file must be exactly the rows the quarantine partition and the
decision table name.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    removed = set(raw["source_row"]) - set(cand["source_row"])
    added   = set(cand["source_row"]) - set(raw["source_row"])
    rd = RD[RD["dataset"] == tag]
    tab = set(rd[rd["quarantine_reasons"].notna()]["source_row"])
    qf  = RUN / f"{tag}_quarantine.csv"
    filep = set(pd.read_csv(qf, dtype=str)["source_row"]) if qf.is_file() else None

    print(f"--- {tag}: raw {len(raw):,} -> candidate {len(cand):,}  removed {len(removed):,}")
    print(f"    decision table names {len(tab):,} quarantined")
    if filep is not None: print(f"    {qf.name} holds {len(filep):,}")
    print(f"    every raw row has a decision row: {set(raw['source_row']) <= set(rd['source_row'])}")

    problems = []
    if added: problems.append(f"{len(added):,} rows in candidate that are not in raw")
    if removed != tab:
        problems.append(f"removed set != decision table: {len(removed ^ tab):,} rows differ")
    if filep is not None and removed != filep:
        problems.append(f"removed set != {qf.name}: {len(removed ^ filep):,} rows differ")
    if not set(raw["source_row"]) <= set(rd["source_row"]):
        problems.append("some raw rows have no row in the decision table")

    D["removed"] = removed
    if problems:
        verdict(tag, "row counts and identity", "FAIL", "\n".join("  "+p for p in problems),
                len(raw), len(raw))
    else:
        verdict(tag, "row counts and identity", "PASS",
                f"{len(removed):,} rows removed ({len(removed)/len(raw):.2%}), and they are "
                f"exactly the rows named by the decision table"
                + (" and the quarantine partition" if filep is not None else "")
                + ". Symmetric difference is zero, not merely equal totals.",
                len(raw), len(raw))

## 2. Quarantine reasons

Every removed row must name the rule that removed it, and the reason must be
present in the row's own content. §9 checks the content; this checks coverage.

In [ ]:
for tag, D in DATASETS.items():
    rd = RD[RD["dataset"] == tag]
    q  = rd[rd["quarantine_reasons"].notna()]
    removed = D["removed"]
    print(f"--- {tag}: {len(q):,} rows carry a reason, {len(removed):,} were removed")
    print(q["quarantine_reasons"].value_counts().head(8).to_string())
    missing = removed - set(q["source_row"])
    holds = rd["hold_reasons"].notna().sum()
    print(f"\n    rows held (hold_reasons): {holds}")
    if missing:
        verdict(tag, "quarantine reasons", "FAIL",
                f"{len(missing):,} removed rows carry no reason.", len(removed), len(removed))
    else:
        verdict(tag, "quarantine reasons", "PASS",
                f"All {len(removed):,} removed rows name a rule. hold_reasons empty on "
                f"all {len(rd):,} rows, matching decision 6 (no automatic whole-row hold).",
                len(removed), len(removed))

## 3. Missingness

Cleaning must not make a column emptier. Measured over the retained rows only,
since removed rows are accounted for separately.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    kept = raw[raw["source_row"].isin(set(cand["source_row"]))]
    shared = [c for c in raw.columns if c in cand.columns and c != "source_row"]
    m = pd.DataFrame({"raw": blank(kept[shared]).mean()*100,
                      "cleaned": blank(cand[shared]).mean()*100})
    m["change_pp"] = m["cleaned"] - m["raw"]
    worse = m[m["change_pp"] > 0.5]
    print(f"--- {tag}: {len(shared)} shared columns over {len(kept):,} retained rows")
    print(m.sort_values("change_pp", ascending=False).head(8).round(2).to_string())
    if worse.empty:
        verdict(tag, "missingness", "PASS",
                "No shared column became more than 0.5pp emptier among retained rows.",
                len(kept), len(cand))
    else:
        verdict(tag, "missingness", "FAIL",
                "Columns that lost data:\n" +
                "\n".join(f"  {c}: +{r['change_pp']:.2f}pp" for c, r in worse.iterrows()),
                len(kept), len(cand))

## 4. Datatype changes

The export re-reads four columns as numbers. The question is not whether the
dtype changed — it did — but whether anything is lost. Two different things get
checked and reported separately, because the first version conflated them:

- **numeric value** preserved, and
- **textual form** preserved.

The second is not preserved and that is worth stating plainly: `5` becomes `5.0`.

In [ ]:
exp_path = PROCESSED / "training_cleaned.csv"
if not exp_path.is_file():
    verdict("training", "datatype changes", "BLOCKED",
            f"No {exp_path.name}; run scripts/export_training_dataset.py --run-dir {RUN}")
else:
    es = pd.read_csv(exp_path, dtype="string", keep_default_na=False, low_memory=False)
    ei = pd.read_csv(exp_path, low_memory=False)
    rows = []
    for c in NUMERIC:
        if c not in es.columns: continue
        txt = es[c].str.strip()
        num = pd.to_numeric(txt.replace({"":None,"null":None}), errors="coerce")
        unparseable = int(((~blank(es[c])) & num.isna()).sum())
        reserialised = ei[c].astype("string").str.strip()
        text_changed = int((txt != reserialised).sum())
        rows.append({"column":c, "dtype":str(ei[c].dtype), "unparseable":unparseable,
                     "text_form_changed":text_changed, "share":text_changed/len(es)})
    t = pd.DataFrame(rows); print(t.to_string(index=False))
    bad = int(t["unparseable"].sum()); changed = int(t["text_form_changed"].sum())
    if bad:
        verdict("training", "datatype changes", "FAIL",
                f"{bad} values do not parse as numbers and become null in the export.",
                len(es), len(es))
    else:
        verdict("training", "datatype changes", "WARN",
                f"No numeric value is lost: every populated value parses. But the textual "
                f"form changes on {changed:,} values across {len(t)} columns - integers "
                f"gain a decimal point (5 -> 5.0). A consumer reading the export as text "
                f"gets different strings than the candidate partition holds. Needs a "
                f"confirmation that the export schema is intended.",
                len(es), len(es))

## 5. Target distributions

Class shares should be close to unchanged. A material shift means rows were
removed unevenly across classes, which biases a model trained on the result.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    worst = ("n/a", 0.0)
    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        r = live(raw[c].str.strip()).value_counts(normalize=True)
        k = live(cand[c].str.strip()).value_counts(normalize=True)
        if r.empty or k.empty: continue
        idx = r.index.union(k.index)
        d = (k.reindex(idx).fillna(0) - r.reindex(idx).fillna(0)).abs().max()*100
        if d > worst[1]: worst = (c, d)
    if worst[1] == 0 and all(live(cand[c]).empty for c in TARGETS if c in cand.columns):
        verdict(tag, "target distributions", "BLOCKED",
                "This dataset carries no labels, so there is no distribution to compare.")
    elif worst[1] < 1:
        verdict(tag, "target distributions", "PASS",
                f"Largest class-share move {worst[1]:.2f}pp ({worst[0]}).", len(raw), len(raw))
    elif worst[1] < 5:
        verdict(tag, "target distributions", "WARN",
                f"{worst[0]} moved {worst[1]:.2f}pp. Check §6 for which class.", len(raw), len(raw))
    else:
        verdict(tag, "target distributions", "FAIL",
                f"{worst[0]} moved {worst[1]:.2f}pp - rows were removed unevenly across classes.",
                len(raw), len(raw))

## 6. Rare-class retention

**Labels with stray whitespace are evaluated, not skipped.** The first version
excluded them by rule, which assumed the answer. A padded label is only noise if
the evidence says so, and the evidence is whether stripping it yields a label
that exists as a real class.

Three outcomes are told apart:

| What happened | How it is recognised |
|---|---|
| Relabelled | the rewrite is recorded in `label_changes.csv` |
| Whitespace variant of a real class | stripping it matches a surviving class |
| Lost | neither of the above |

In [ ]:
relabel = {}
if len(LABEL_CHANGES):
    for _, r in LABEL_CHANGES.iterrows():
        relabel.setdefault((r.get("dataset"), r["column"], r["old_value"]), []).append(
            (r["new_value"], r.get("rule","")))

for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    findings = []
    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        rv = live(raw[c]); kv = live(cand[c])
        if rv.empty: continue
        surviving = set(kv.str.strip())
        gone = [l for l in rv.unique() if l not in set(kv)]
        if not gone: continue
        print(f"\n--- {tag} / {c}: {len(gone)} label(s) in raw and not in candidate")
        for lbl in gone:
            n = int((rv == lbl).sum()); stripped = lbl.strip()
            rec = relabel.get((tag, c, lbl))
            if rec:
                print(f"  RELABELLED {lbl!r} ({n}) -> {sorted({d for d,_ in rec})} "
                      f"[{sorted({r for _,r in rec if r})}] {len(rec)} row(s)")
                if len(rec) < n:
                    findings.append(f"{c}: {lbl!r} had {n} rows, only {len(rec)} relabels recorded")
            elif stripped != lbl and stripped in surviving:
                grew = int((kv.str.strip()==stripped).sum()) - int((rv.str.strip()==stripped).sum()-n)
                print(f"  WHITESPACE VARIANT {lbl!r} ({n}) of surviving {stripped!r} (net {grew:+d})")
            elif stripped != lbl:
                print(f"  PADDED, NO REAL COUNTERPART {lbl!r} ({n}) - stripping gives "
                      f"{stripped[:40]!r}, which is not a class in this target")
            else:
                print(f"  LOST {lbl!r} ({n} rows) - no relabel recorded, no variant match")
                findings.append(f"{c}: {lbl!r} lost with {n} rows")

    # classes that survived but shrank
    shrunk = []
    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        r = live(raw[c].str.strip()).value_counts(); k = live(cand[c].str.strip()).value_counts()
        for lbl, n in k.items():
            if lbl in r.index and r[lbl] and 1 - n/r[lbl] > 0.20:
                shrunk.append((c, lbl, int(r[lbl]), int(n)))
    print(f"\n{tag}: surviving classes that lost >20% of their rows: {len(shrunk)}")
    for c, lbl, a, b in shrunk: print(f"    {c} {lbl!r}: {a} -> {b}")

    pop = len(live(raw[[c for c in TARGETS if c in raw.columns]].stack())) or len(raw)
    if findings:
        verdict(tag, "rare-class retention", "FAIL",
                "Labels lost with no recorded relabel and no whitespace counterpart:\n" +
                "\n".join("  "+f for f in findings), len(raw), len(raw))
    elif shrunk:
        verdict(tag, "rare-class retention", "WARN",
                f"No class lost outright, but {len(shrunk)} lost over 20% of their rows.",
                len(raw), len(raw))
    else:
        verdict(tag, "rare-class retention", "PASS",
                "Every label absent from the candidate file is either a recorded relabel or a "
                "whitespace-padded string with no real counterpart. No class lost outright, "
                "and no surviving class lost more than 20% of its rows.", len(raw), len(raw))

## 7. Feature values on retained rows

**The check the first version got wrong.** It compared 76,219 rows, reported
PASS, and the document repeated that number while 83,938 rows had been retained.
The gap was never explained because it was never noticed.

This version joins on `source_row`, prints the coverage next to the population,
and the recorder refuses to call it PASS if the two differ.

Any value that changed is a finding unless the decision log or the handoff README
approves the transform. Unapproved changes are **FAIL**.

In [ ]:
APPROVED = {
    # column -> (what the transform does, where it is approved)
    "ProductCategory": ("separator spacing and exact-duplicate path collapse",
                        "handoff README: 'safe spacing/category formatting'"),
}

for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    feats = [c for c in raw.columns
             if c in cand.columns and c not in TARGETS and c != "source_row"]
    j = (raw.set_index("source_row")[feats]
            .join(cand.set_index("source_row")[feats], lsuffix="_r", rsuffix="_c", how="inner"))
    print(f"--- {tag}: compared {len(j):,} rows of {len(cand):,} retained, "
          f"over {len(feats)} feature columns")
    if len(j) != len(cand):
        print(f"    ! {len(cand)-len(j):,} retained rows could not be joined")

    changed, unapproved = [], []
    for c in feats:
        d = j[c+"_r"] != j[c+"_c"]
        if not d.any(): continue
        changed.append((c, int(d.sum()), d.mean()))
        print(f"\n  {c}: {d.sum():,} rows changed ({d.mean():.2%})")
        for _, r in j.loc[d, [c+"_r", c+"_c"]].head(2).iterrows():
            print(f"      {str(r.iloc[0])[:62]!r}\n   -> {str(r.iloc[1])[:62]!r}")
        if c in APPROVED:
            what, where = APPROVED[c]
            # verify the change really is only the approved transform
            if c == "ProductCategory":
                norm = lambda s: re.sub(r"\s*>\s*", ">", str(s)).strip()
                dedup = lambda s: ">".join(dict.fromkeys(norm(s).split(">")))
                ok = (j.loc[d, c+"_r"].map(dedup) == j.loc[d, c+"_c"].map(dedup))
                print(f"      approved as: {what} ({where})")
                print(f"      rows explained by that transform: {ok.sum():,} / {int(d.sum()):,}")
                if not ok.all():
                    unapproved.append(f"{c}: {int((~ok).sum())} rows change beyond {what}")
            else:
                print(f"      approved as: {what} ({where})")
        else:
            unapproved.append(f"{c}: {int(d.sum()):,} rows changed, no approving rule found")

    if unapproved:
        verdict(tag, "feature values on retained rows", "FAIL",
                "Changes with no approving rule:\n" + "\n".join("  "+u for u in unapproved),
                len(j), len(cand))
    elif changed:
        verdict(tag, "feature values on retained rows", "WARN",
                "Values changed in " + ", ".join(f"{c} ({n:,} rows)" for c,n,_ in changed) +
                ". Each is explained by a transform the handoff README documents, but none "
                "of them appears in docs/decisions.md. They need a recorded decision rather "
                "than a fix.", len(j), len(cand))
    else:
        verdict(tag, "feature values on retained rows", "PASS",
                "No feature value changed on any retained row.", len(j), len(cand))

## 8. Malformed labels

Whitespace-padded values in the target columns, raw against candidate.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    rows = []
    for c in TARGETS:
        if c not in cand.columns: continue
        rb = int((live(raw[c]) != live(raw[c]).str.strip()).sum()) if c in raw.columns else 0
        cb = int((live(cand[c]) != live(cand[c]).str.strip()).sum())
        rows.append({"target": c, "raw": rb, "candidate": cb})
    t = pd.DataFrame(rows); print(f"--- {tag}"); print(t.to_string(index=False))
    left, found = int(t["candidate"].sum()), int(t["raw"].sum())
    if left:
        verdict(tag, "malformed labels", "FAIL",
                f"{left} padded value(s) survive and will be treated as classes downstream.",
                len(cand), len(cand))
    else:
        verdict(tag, "malformed labels", "PASS",
                f"All {found} padded values handled; none survives.", len(raw), len(raw))

## 9. Structural integrity of retained rows

Rewritten after review. The first version ran two cheap signals and concluded
there was no corruption, which the evidence did not support.

**Quoted records are included.** They were the ones most likely to be displaced,
and excluding them is how a displacement audit misses displacement.

**Each column is checked against its own domain** — numeric columns must hold
numbers, URL columns must hold URLs, the timestamp must not hold a URL. A value
in the wrong column is the actual signature of a shifted row.

**Shared vocabulary between targets is reported, not assumed to be corruption.**
A Platform called `Allergy` and a Segment called `Allergy` are different things
with the same name.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    kept_src = set(cand["source_row"])
    quoted_raw = int(raw["ProductName"].str.contains('"', na=False).sum())
    quoted_kept = int(cand["ProductName"].str.contains('"', na=False).sum())
    print(f"--- {tag}: {quoted_raw:,} raw rows contain a quote in ProductName; "
          f"{quoted_kept:,} of them are retained and included below")

    probs = {}
    for c in NUMERIC:
        if c in cand.columns:
            probs[f"{c} not numeric"] = int((~(cand[c].str.strip().str.match(r"^-?\d+(\.\d+)?$")
                                               | blank(cand[c]))).sum())
    for c in URLCOLS:
        if c in cand.columns:
            v = cand[c].str.strip().str.replace('"', '', regex=False).str.lower()
            probs[f"{c} not a URL"] = int(((~blank(cand[c])) & ~v.str.startswith(("http://","https://"))).sum())
    if "MDM_InsertDateTime" in cand.columns:
        ts = cand["MDM_InsertDateTime"].str.strip()
        probs["MDM_InsertDateTime holds a URL"] = int(ts.str.contains("http", case=False, na=False).sum())
        probs["MDM_InsertDateTime not numeric"] = int((~(ts.str.match(r"^\d+(\.\d+)?$")
                                                          | blank(cand["MDM_InsertDateTime"]))).sum())
    ragged = 0
    with open(DATASETS[tag]["run"].parent.parent/"provided"/
              ("Selfcare_Training_data (1).csv" if tag=="training" else "Selfcare_Target_data (1).csv"),
              encoding="utf-8", errors="ignore") as f:
        head = f.readline().count(",")
    for k, v in probs.items(): print(f"    {'OK     ' if v==0 else 'REVIEW '} {k:38s} {v}")

    vocab = {c: set(live(cand[c].str.strip())) for c in TARGETS if c in cand.columns}
    overlaps = []
    cols = list(vocab)
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            for val in sorted(vocab[a] & vocab[b]):
                overlaps.append((val, a, int((cand[a].str.strip()==val).sum()),
                                      b, int((cand[b].str.strip()==val).sum())))
    print(f"\n    values shared between two targets: {len(overlaps)}")
    for val, a, na, b, nb in overlaps:
        print(f"      {val!r}: {a} ({na:,}) and {b} ({nb:,})")

    bad = {k: v for k, v in probs.items() if v}
    if bad:
        verdict(tag, "structural integrity", "FAIL",
                "Values sitting in the wrong column among RETAINED rows:\n" +
                "\n".join(f"  {k}: {v:,}" for k, v in bad.items()), len(cand), len(cand))
    else:
        verdict(tag, "structural integrity", "PASS",
                f"Every retained row holds values of the right kind in every checked column, "
                f"including the {quoted_kept:,} that contain a quote. "
                + (f"{len(overlaps)} value(s) appear in two targets; they are distinct "
                   f"concepts sharing a name, not displacement." if overlaps else ""),
                len(cand), len(cand))

## 10. What left, and what the evidence actually shows

Corrected after review. The first version wrote that rows with no product text
were corrupted. **Missing text is not evidence of corruption.** It is the
condition the pipeline's own rule names, and the row may simply be sparse.

This section reports what each removed row has, and separates what is
*demonstrated* from what the rule *asserts*.

In [ ]:
for tag, D in DATASETS.items():
    raw, removed = D["raw"], D["removed"]
    if not removed:
        verdict(tag, "information loss", "PASS", "No rows removed."); continue
    gone = raw[raw["source_row"].isin(removed)]
    rd = RD[RD["dataset"] == tag].set_index("source_row")
    gone = gone.join(rd[["quarantine_reasons"]], on="source_row")

    notext = (blank(gone["ProductName"]) & blank(gone["ProductDescription"])
              & blank(gone["ProductContents"]))
    isnum = lambda s: s.str.strip().str.match(r"^-?\d+(\.\d+)?$") | blank(s)
    displaced = sum([(~isnum(gone[c])).astype(int) for c in NUMERIC]).astype(bool)
    urlts = gone["MDM_InsertDateTime"].str.contains("http", case=False, na=False)

    print(f"--- {tag}: {len(gone):,} removed")
    print(f"    demonstrable displacement (value in the wrong column): {int((displaced|urlts).sum()):,}")
    print(f"    no product text at all                              : {int(notext.sum()):,}")
    print(f"    neither                                             : {int((~(displaced|urlts) & ~notext).sum()):,}")
    labelled = (~blank(gone[[c for c in TARGETS if c in gone.columns]])).any(axis=1)
    print(f"    carried at least one label                          : {int(labelled.sum()):,}")

    verdict(tag, "information loss", "PASS" if not int((~(displaced|urlts) & ~notext).sum()) else "WARN",
            f"{int((displaced|urlts).sum()):,} removed rows show a value sitting in the wrong "
            f"column, which is demonstrable displacement. A further {int(notext.sum()):,} have no "
            f"product text; that is the condition rule 4 names, and it is NOT by itself evidence "
            f"that the row is corrupted - it may be a sparse record. Whether such rows should be "
            f"dropped is a product decision, and the rule records it as made. "
            f"{int(labelled.sum()):,} of the removed rows carried a label.",
            len(gone), len(gone))

## 11. Summary

In [ ]:
s = pd.DataFrame(VERDICTS)
if len(s):
    order = {"FAIL":0,"WARN":1,"BLOCKED":2,"PASS":3}
    s = s.sort_values(["status","dataset"], key=lambda col: col.map(order) if col.name=="status" else col)
    print(s[["dataset","criterion","status","covered","population"]].to_string(index=False))
    print()
    c = s["status"].value_counts()
    print(f"FAIL {c.get('FAIL',0)}   WARN {c.get('WARN',0)}   "
          f"BLOCKED {c.get('BLOCKED',0)}   PASS {c.get('PASS',0)}")
    print()
    for _, r in s[s["status"] != "PASS"].iterrows():
        print(f"[{r['status']}] {r['dataset']} / {r['criterion']}")
        for line in str(r["detail"]).split("\n"): print("    " + line)
        print()
    out = DATA/"interim"; out.mkdir(parents=True, exist_ok=True)
    s.to_csv(out/"cleaning_audit_summary.csv", index=False)
    print("written to", out/"cleaning_audit_summary.csv")
    under = s[(s["status"]=="PASS") & s["covered"].notna() & s["population"].notna()
              & (s["covered"] < s["population"])]
    print("\nPASS verdicts with incomplete coverage:", len(under), "(must be 0)")
else:
    print("no verdicts - run from the top")